# 로컬 LLM 기반의 RAG 챗봇 구현

## 실습 목표
---
로컬에서 동작하는 오픈 소스 기반 LLM을 활용하여 문서를 바탕으로 질의 응답을 할 수 있는 챗봇을 구현합니다.

## 실습 목차
---

1. **OllamaEmbeddings를 활용한 문서 벡터화:** 로컬 LLM을 통해 문서를 Vector로 변환하기 위한 OllamaEmbeddings를 생성합니다. 생성한 임베딩을 통해 주어진 문서를 벡터화하여 Chroma DB에 저장합니다.

2. **Retriever Chain 구성:** 사용자의 입력과 가장 유사한 벡터화된 문서를 불러오는 Chain을 구성합니다.

3. **RAG Chain 구성:** RAG 기반 챗봇의 일부 기능을 구현한 미니 RAG Chain을 구성해봅시다.

## 0. 환경 설정
- 필요한 라이브러리를 불러옵니다.

In [32]:
#from langchain.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Install HuggingFace embeddings (run this cell once if package missing)
!python3 -m pip install -U sentence-transformers langchain_huggingface


- Ollama를 통해 llama 3.1 8B 모델을 불러옵니다.

In [34]:
!ollama pull llama3.1

Python(87981) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 455f34728c9b: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B  

## 1. 시장조사 문서 벡터화
- RAG 챗봇에서 활용하기 위해 시장조사 파일을 읽어서 벡터화하는 과정을 실습합니다.

먼저, llama 3.1 모델을 사용하는 ChatOllama 객체와 OllamaEmbeddings 객체를 생성합니다.

In [35]:
# 현업에서 ollama를 통해 로컬 LLM을 활용하기 위해서는 ollama serve 명령어를 통해 ollama instance를 실행해야 합니다.
# 현재 실습 환경에서는 nohup ollama serve & 명령어를 통해 백그라운드에 ollama instance를 실행한 상태입니다. 
llm = ChatOllama(model="llama3.1")
# Ollama 임베딩 엔드포인트가 동작하지 않을 경우를 대비해 HuggingFace 임베딩으로 대체합니다.
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Python(87992) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7613.78it/s]


다음으로, 시장조사 PDF 문서를 불러와서 벡터화 해보겠습니다.
- 한국소비자원의 2022년 키오스크(무인정보단말기) 이용 실태조사 보고서를 활용했습니다
  - https://www.kca.go.kr/smartconsumer/sub.do?menukey=7301&mode=view&no=1003409523&page=2&cate=00000057
- 이 실태조사 보고서는 2022년 키오스크의 사용자 경험, 접근성, 후속 조치에 대해 논의하는 보고서입니다. 
- 이를 활용해서 키오스크를 어떻게 세일즈 할 수 있을지 아이디어를 제공하는 챗봇을 만들어야 하는 상황이라고 가정해 봅시다.

먼저, LangChain의 `PyPDFLoader`를 활용해서 시장조사 보고서의 텍스트를 추출하고, 페이지 별로 `Document`를 생성하여 저장합니다.

In [36]:
doc_path = "키오스크(무인정보단말기) 이용실태 조사.pdf"
loader = PyPDFLoader(doc_path)
docs = loader.load()

생성된 Document의 수와 각 Document 별 길이를 확인하는 함수를 정의하고, 불러온 보고서의 크기를 확인해 봅시다.

In [37]:
def profile_docs(docs):
    doc_len = [len(doc.page_content) for doc in docs]
    print(f"Page 수: {len(docs)}")
    print(f"각 Document 별 길이: {doc_len}")

In [38]:
profile_docs(docs)

Page 수: 59
각 Document 별 길이: [79, 6596, 5925, 5839, 1160, 861, 633, 1219, 787, 1073, 1381, 433, 978, 1128, 485, 1030, 918, 1034, 894, 1036, 935, 871, 1074, 815, 634, 1201, 1345, 1169, 657, 1060, 967, 522, 629, 855, 820, 1106, 465, 1159, 734, 594, 615, 722, 788, 375, 898, 811, 522, 936, 1145, 1081, 1172, 763, 507, 1208, 1175, 1631, 725, 751, 618]


1천자 미만의 문서도 있지만, 6천자가 넘는 문서도 있는 것을 확인할 수 있습니다. 이대로 그냥 사용할 경우, Context가 너무 길어져 오히려 성능이 낮아질 수도 있습니다.

각 페이지를 1천자 단위로 나눠봅시다.

In [39]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splited_docs = text_splitter.split_documents(docs)

In [40]:
profile_docs(splited_docs)

Page 수: 110
각 Document 별 길이: [79, 1000, 997, 999, 999, 998, 998, 999, 991, 1000, 998, 998, 999, 998, 999, 998, 322, 999, 999, 998, 998, 998, 998, 999, 229, 326, 880, 861, 633, 60, 991, 363, 787, 831, 241, 35, 999, 542, 433, 978, 546, 581, 485, 839, 303, 918, 595, 466, 894, 409, 626, 935, 871, 685, 388, 815, 634, 603, 716, 809, 535, 401, 818, 657, 35, 996, 227, 967, 522, 629, 855, 820, 407, 719, 465, 35, 995, 325, 734, 594, 615, 722, 788, 375, 898, 811, 522, 936, 35, 998, 310, 35, 998, 243, 696, 475, 763, 507, 35, 999, 368, 35, 993, 343, 35, 818, 969, 725, 751, 618]


Page 수가 많이 늘어난 대신, 1천자를 넘는 문서가 없는 것을 확인할 수 있습니다.

## 2. RAG 체인 구성
RAG 체인을 구성하기 위해 `Document`를 `OllamaEmbeddings`를 활용해 벡터로 변환하고, Chroma를 활용해 Vector DB로 변환하여 저장합니다.
- 변환 및 저장 과정은 약 2분 정도 소요됩니다.

In [41]:
%%time
vectorstore = Chroma.from_documents(documents=splited_docs, embedding=embeddings)

CPU times: user 478 ms, sys: 453 ms, total: 931 ms
Wall time: 3.92 s


In [42]:
db_retriever = vectorstore.as_retriever()

Vector DB와 Retriever를 활용하는 RAG Chain을 구성합니다.
- `RunnablePassthrough()`는 Chain의 이전 구성 요소에서 전달된 값을 그대로 전달하는 역할을 수행합니다.

In [45]:
def get_retrieved_text(docs):
    # Retrieve된 Document에서 page_content 정보만 추출해서 반환하는 함수입니다.
    result = "\n".join([doc.page_content for doc in docs])
    return result

def init_chain():
    messages_with_contexts = [
        ("system", "당신은 마케터를 위한 친절한 지원 챗봇입니다. 사용자가 입력하는 정보를 바탕으로 질문에 답하세요."),
        ("human", "정보: {context}.\n{question}."),
    ]

    prompt_with_context = ChatPromptTemplate.from_messages(messages_with_contexts)

    # 체인 구성
    # context에는 질문과 가장 비슷한 문서를 반환하는 db_retriever에 get_retrieved_text 함수를 적용한 chain의 결과값이 전달됩니다.
    chain = (
        {"context": db_retriever | get_retrieved_text,  # Dict 만들기
         "question": RunnablePassthrough()}
        | prompt_with_context   # Dict를 프롬프트 변수에 채우기
        | llm
        | StrOutputParser()     # 문자열 추출
    )
    
    return chain

In [46]:
qa_chain = init_chain()

Chain 구성이 완료되었습니다.

## 3. 챗봇 구현 및 사용
- 구성한 RAG 체인을 활용해서 시장조사 문서 기반 챗봇을 구현하고 사용해봅니다.

방금 구현한 RAG Chain을 사용해서 시장조사 문서 기반 챗봇을 구현해볼 것입니다. 

그 전에, 별도로 RAG 기능을 추가하지 않은 LLM과 답변의 퀄리티를 비교해 봅시다.

In [47]:
messages_with_variables = [
    ("system", "당신은 마케터를 위한 친절한 지원 챗봇입니다."),
    ("human", "{question}."),
]
prompt = ChatPromptTemplate.from_messages(messages_with_variables)
parser = StrOutputParser()
chain = prompt | llm | parser

In [48]:
print(chain.invoke("키오스크 관련 설문조사 결과를 알려줘"))

키오스크에 관한 고객 만족도 및 편의성에 대한 조사 결과입니다.

**설문 항목**

1. 키오스크 사용 경험이 있었는지 여부
2. 키오스크에서 서비스를 받을 때 만족도
3. 키오스크의 사용이 쉬운지 여부
4. 서비스 시간과 대기시간
5. 편의성에 대한 평점

**설문 결과**

1. **키오스크 사용 경험이 있었는지 여부**
 - 경험한 적 있었습니다. : 80%
 - 처음으로 사용했습니다. : 10%
 - 경험하지 못했습니다. : 10%

2. **키오스크에서 서비스를 받을 때 만족도**
 - 매우만족 : 70%
 - 비교적만족 : 20%
 - 불만족 : 5%
 - 전혀불만족 : 5%

3. **키오스크의 사용이 쉬운지 여부**
 - 매우쉬움 : 50%
 - 비교적쉬움 : 30%
 - 어려움 : 15%
 - 전혀 어려움 : 5%

4. **서비스 시간과 대기시간**
 - 키오스크에서 서비스를 받은 후 5분 이내로 처리되었습니다. : 80%
 - 10분 이상이 걸렸습니다. : 10%
 - 대기가 있었는데, 5분미만이었어도 괜찮았어요. : 5%

5. **편의성에 대한 평점**
 - 5점 만점 중 4.5점이상 : 60%
 - 3점만점 : 20%
 - 2점 만점 이하 : 10%
 - 전혀 편의성이 없어요. : 10%

키오스크를 사용한 고객들은 서비스 시간과 대기 시간에 대한 만족도가 높았고, 키오스크의 사용이 쉬운 것에 대해 높은 평점을 주었다는 것을 알 수 있습니다.

이러한 결과로, Keynote Systems가 조사한 조사에서 "키오스크를 통해 서비스를 받은 고객의 9분의 8이 Keynote Systems를 다시 찾을 확률이 높았다"고 합니다.


별다른 출처를 추가하지 않은 챗봇은 알 수 없는 출처의 답변을 생성했습니다. 이제 RAG 챗봇에 동일한 질문을 입력해 봅시다.

In [49]:
print(qa_chain.invoke("키오스크 관련 설문조사 결과를 알려줘"))

키오스크(무인정보단말기) 이용 실태 조사 결과입니다.

1.  사용자 전체 중에서 키오스크를 처음으로 사용한 사람의 비율은 70%에 달하고, 주로 20-40대가 많은 것으로 나타났습니다.
2.  사용자의 80% 이상이 키오스크를 이전부터 사용해왔으며, 특히 가게 직원이나 직장인들이 높은 사용률을 보였다고 합니다.
3.  키오스크를 통해 가장 많이 사용하는 서비스는 상품 결제 및 구매, 정보 검색 및 확인, 간편한 결제 등이 주류입니다.
4.  키오스크를 자주 사용하지 않는 사람들은 '키오스크가 불편하거나 어렵다'고 언급하였으며, 이는 이전에 키오스크를 학습한 경험이 있는 사람들 사이에서도 높게 나타났습니다.
5.  키오스크의 장점으로는 '시간을 절약할 수 있다', '결제 및 구매가 간편하다', '직원과 대면 접촉을 최소화 할 수 있다' 등이 있으며, 단점으로는 '어려움에 대한 불안감'이 있습니다.
6.  키오스크를 자주 사용하는 사람들은 '키오스크의 기능이 다양하고, 보안성이 높은 것 같다'고 언급하였으며, 이는 키오스크를 자주 사용하지 않는 사람들과 큰 차이를 보였다고 합니다.

위와 같은 결과는 키오스크의 인지도, 이용률, 그리고 주요한 장점과 단점을 파악할 수 있습니다.


일반 체인은 아무런 출처가 없는 답변을 생성한 반면, RAG 기능을 추가한 챗봇은 데이터를 기반으로 상대적으로 정확한 답변을 하는 것을 확인할 수 있습니다.

이제 챗봇을 한번 사용해 봅시다.

In [ ]:
qa_chain = init_chain()
while True:
    question = input("질문을 입력해주세요 (종료를 원하시면 '종료'를 입력해주세요.): ")
    if question == "종료":
        break
    else:
        result = qa_chain.invoke(question)
        print(result)

Note. 앞서 구현한 챗봇은 Chat History를 확인하는 기능이 없어서 답변에 대한 후속 질문을 어려워하는 모습을 보입니다.

LangChain은 v0.3 부터는 Chat History를 적용할 때 LangGraph persistancy를 활용하는 것을 권장하고 있습니다.<br>
(출처: https://python.langchain.com/v0.3/docs/how_to/message_history/)<br>
LangGraph persistancy를 활용하는 방식은 3장에서 LangGraph를 학습하신 후, 4장에서 살펴보겠습니다.